In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.5 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [4]:
# model_name = "microsoft/deberta-v3-small"
# save_dir = "/kaggle/input/notebooks/ayanansari111/deberta-finetuned/deberta_finetuned"

# base_model = AutoModelForMultipleChoice.from_pretrained(model_name)
# tokenizer = AutoTokenizer.from_pretrained(save_dir)

# model = PeftModel.from_pretrained(base_model, save_dir)
# model.eval()

In [5]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType

train_subset = train.copy()

model_name = "microsoft/deberta-v3-small"
dtokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForMultipleChoice.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
deberta = get_peft_model(base_model, lora_config)

raw_dataset = Dataset.from_pandas(train)

label_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
options_keys = ["A", "B", "C", "D", "E"]

def preprocess_function(examples):
    batch_input_ids = []
    batch_attention_mask = []
    
    for i in range(len(examples["prompt"])):
        prompt = examples["prompt"][i]
        formatted_opts = [str(prompt) + " [SEP] " + str(examples[opt][i]) for opt in options_keys]
        
        tokenized = dtokenizer(
            formatted_opts,
            padding="max_length",
            truncation=True,
            max_length=64 
        )
        batch_input_ids.append(tokenized["input_ids"])
        batch_attention_mask.append(tokenized["attention_mask"])
        
    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": [label_mapping[ans] for ans in examples["answer"]]
    }

dprocessed_dataset = raw_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=raw_dataset.column_names
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-4,
    per_device_train_batch_size=4,       
    gradient_accumulation_steps=1,       
    max_steps=1000,                         
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=deberta,
    args=training_args,
    train_dataset=dprocessed_dataset
)

train_output = trainer.train()

print(f"Global Step: {train_output.global_step}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

... (3 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,3.230059
200,3.129199
300,2.678389
400,2.345654
500,1.930342
600,1.787124
700,1.588154
800,1.425525
900,1.441233
1000,1.387319


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Global Step: 1000


# Q1

In [6]:
import torch.nn.functional as F
num = 25
first_item = dprocessed_dataset[num]

model_inputs = {
    "input_ids": torch.tensor([first_item['input_ids']]).to(deberta.device),
    "attention_mask": torch.tensor([first_item['attention_mask']]).to(deberta.device)
}

deberta.eval()
with torch.no_grad():
    output = deberta(**model_inputs)
    
logits = output.logits
dprobs = F.softmax(logits,dim=-1)
print(dprobs)
print(train.iloc[num])

tensor([[0.0085, 0.0202, 0.0061, 0.0178, 0.9473]], device='cuda:0',
       dtype=torch.float16)
id                                                       26
prompt    Choose the correct answer: What is Hesse's pri...
A         Hesse's principle of transfer is a concept in ...
B         Hesse's principle of transfer is a concept in ...
C         Hesse's principle of transfer is a concept in ...
D         Hesse's principle of transfer is a concept in ...
E         Hesse's principle of transfer is a concept in ...
answer                                                    E
Name: 25, dtype: object


# Finetuning Roberta

In [7]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType

train = train.copy()

model_name = "roberta-base"
rtokenizer = AutoTokenizer.from_pretrained(model_name)
rbase_model = AutoModelForMultipleChoice.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
roberta = get_peft_model(rbase_model, lora_config)

raw_dataset = Dataset.from_pandas(train)

label_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
options_keys = ["A", "B", "C", "D", "E"]

def preprocess_function(examples):
    batch_input_ids = []
    batch_attention_mask = []
    
    for i in range(len(examples["prompt"])):
        prompt = examples["prompt"][i]
        formatted_opts = [str(prompt) + " [SEP] " + str(examples[opt][i]) for opt in options_keys]
        
        tokenized = rtokenizer(
            formatted_opts,
            padding="max_length",
            truncation=True,
            max_length=64 
        )
        batch_input_ids.append(tokenized["input_ids"])
        batch_attention_mask.append(tokenized["attention_mask"])
        
    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": [label_mapping[ans] for ans in examples["answer"]]
    }

rprocessed_dataset = raw_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=raw_dataset.column_names
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-4,
    per_device_train_batch_size=4,       
    gradient_accumulation_steps=1,       
    max_steps=1000,                         
    logging_steps=100,
    report_to="none",
)

rtrainer = Trainer(
    model=roberta,
    args=training_args,
    train_dataset=rprocessed_dataset
)

train_output = rtrainer.train()

print(f"Global Step: {train_output.global_step}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

... (3 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,3.220324
200,2.899720
300,2.490574
400,2.043837
500,1.788236
600,1.718834
700,1.445412
800,1.253527
900,1.299668
1000,1.265185


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Global Step: 1000


In [8]:
import torch.nn.functional as F
num = 25
first_item = rprocessed_dataset[num]

model_inputs = {
    "input_ids": torch.tensor([first_item['input_ids']]).to(roberta.device),
    "attention_mask": torch.tensor([first_item['attention_mask']]).to(roberta.device)
}

roberta.eval()
with torch.no_grad():
    output = roberta(**model_inputs)
    
logits = output.logits
rprobs = F.softmax(logits,dim=-1)
print(rprobs)
print(train.iloc[num])

tensor([[0.0120, 0.0112, 0.0124, 0.0123, 0.9521]], device='cuda:0')
id                                                       26
prompt    Choose the correct answer: What is Hesse's pri...
A         Hesse's principle of transfer is a concept in ...
B         Hesse's principle of transfer is a concept in ...
C         Hesse's principle of transfer is a concept in ...
D         Hesse's principle of transfer is a concept in ...
E         Hesse's principle of transfer is a concept in ...
answer                                                    E
Name: 25, dtype: object


# Q2

In [9]:
avg_probs = (rprobs + dprobs)/2
avg_probs

tensor([[0.0103, 0.0157, 0.0093, 0.0151, 0.9497]], device='cuda:0')

# Q3

In [10]:
w_probs = (0.7*dprobs) + (0.3*rprobs)
w_probs

tensor([[0.0095, 0.0175, 0.0080, 0.0162, 0.9487]], device='cuda:0')

# Q4

In [11]:
import torch
pred_idx = torch.topk(w_probs,k=3).indices
pred_idx

tensor([[4, 1, 3]], device='cuda:0')

# Q6

In [12]:
train_subset = train.head(50)
raw_subset = Dataset.from_pandas(train_subset)

dprocessed_dataset_50 = raw_subset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=raw_subset.column_names
)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [13]:
import torch.nn.functional as F
import torch
dpreds = []
options = ['A','B','C','D','E']
for i in range(50):
    data_item = dprocessed_dataset_50[i]
    input_ids = torch.tensor(data_item['input_ids']).unsqueeze(0).to(deberta.device)
    attention_mask = torch.tensor(data_item['attention_mask']).unsqueeze(0).to(deberta.device)    
    deberta.eval()
    with torch.no_grad():
        output = deberta(input_ids = input_ids,attention_mask = attention_mask)
    logits = output.logits
    probs = F.softmax(logits,dim=1)
    top1_idx = torch.topk(probs,k=1).indices
    ans_label = options[top1_idx]
    dpreds.append(ans_label)

In [14]:
def aug_preprocess_function(examples):
    batch_input_ids = []
    batch_attention_mask = []
    
    for i in range(len(examples["prompt"])):
        prompt = examples["prompt"][i]
        augmentation = "Answer the following multiple-choice question carefully:"
        aug_prompt = str(prompt) + str(augmentation)
        formatted_opts = [str(aug_prompt) + " [SEP] " + str(examples[opt][i]) for opt in options_keys]
        
        tokenized = dtokenizer(
            formatted_opts,
            padding="max_length",
            truncation=True,
            max_length=64 
        )
        batch_input_ids.append(tokenized["input_ids"])
        batch_attention_mask.append(tokenized["attention_mask"])
        
    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": [label_mapping[ans] for ans in examples["answer"]]
    }

In [15]:
aug_dprocessed_dataset = raw_subset.map(
    aug_preprocess_function, 
    batched=True, 
    remove_columns=raw_subset.column_names
)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [16]:
import torch.nn.functional as F
import torch
aug_dpreds = []
options = ['A','B','C','D','E']
for i in range(50):
    data_item = aug_dprocessed_dataset[i]
    input_ids = torch.tensor(data_item['input_ids']).unsqueeze(0).to(deberta.device)
    attention_mask = torch.tensor(data_item['attention_mask']).unsqueeze(0).to(deberta.device)    
    deberta.eval()
    with torch.no_grad():
        output = deberta(input_ids = input_ids,attention_mask = attention_mask)
    logits = output.logits
    probs = F.softmax(logits,dim=1)
    top1_idx = torch.topk(probs,k=1).indices
    ans_label = options[top1_idx]
    aug_dpreds.append(ans_label)

In [17]:
differences = sum(a != b for a, b in zip(dpreds, aug_dpreds))
differences

35

# Q7 & Q8

In [18]:
roberta.device

device(type='cuda', index=0)

In [19]:
import torch
from torch.utils.data import DataLoader
from transformers import default_data_collator
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deberta.to(device)
roberta.to(device)

deberta.eval()
roberta.eval()

batch_size = 16

ddataset_100 = dprocessed_dataset.select(range(100))
rdataset_100 = rprocessed_dataset.select(range(100))

dloader = DataLoader(
    ddataset_100,
    batch_size=batch_size,
    collate_fn=default_data_collator,
    shuffle=False
)

rloader = DataLoader(
    rdataset_100,
    batch_size=batch_size,
    collate_fn=default_data_collator,
    shuffle=False
)

deberta_preds = []
ensemble_preds = []
deberta_conf = []
ensemble_conf = []

with torch.no_grad():

    for dbatch, rbatch in zip(dloader, rloader):

        dbatch = {k: v.to(device) for k, v in dbatch.items()}
        rbatch = {k: v.to(device) for k, v in rbatch.items()}

        d_logits = deberta(**dbatch).logits
        r_logits = roberta(**rbatch).logits

        d_probs = F.softmax(d_logits,dim=1)
        r_probs = F.softmax(r_logits,dim=1)

        d_top1 = torch.argmax(d_probs, dim=1)

        ensemble_probs = 0.3 * d_probs + 0.7 * r_probs

        ensemble_top1 = torch.argmax(ensemble_probs, dim=1)

        d_confidence = d_probs.max(dim=1).values
        ensemble_confidence = ensemble_probs.max(dim=1).values

        deberta_conf.extend(d_confidence.cpu().tolist())
        ensemble_conf.extend(ensemble_confidence.cpu().tolist())
        deberta_preds.extend(d_top1.cpu().tolist())
        ensemble_preds.extend(ensemble_top1.cpu().tolist())

# Count differing predictions
different = sum(
    d != e
    for d, e in zip(deberta_preds, ensemble_preds)
)

# Postive confidence gain
confidence_gain = [
    e - d
    for d, e in zip(deberta_conf, ensemble_conf)
]
positive_gain = sum(gain > 0 for gain in confidence_gain)

print("Rows with positive confidence gain:", positive_gain)
print("Rows with different Top-1 predictions:", different)

Rows with positive confidence gain: 61
Rows with different Top-1 predictions: 13


In [20]:
deberta_top3 = []
ensemble_top3 = []

with torch.no_grad():

    for dbatch, rbatch in zip(dloader, rloader):

        dbatch = {k: v.to(device) for k, v in dbatch.items()}
        rbatch = {k: v.to(device) for k, v in rbatch.items()}

        d_probs = torch.softmax(deberta(**dbatch).logits, dim=1)
        r_probs = torch.softmax(roberta(**rbatch).logits, dim=1)

        ensemble_probs = 0.3 * d_probs + 0.7 * r_probs

        d_top3 = torch.topk(d_probs, k=3, dim=1).indices
        e_top3 = torch.topk(ensemble_probs, k=3, dim=1).indices

        deberta_top3.extend(d_top3.cpu().tolist())
        ensemble_top3.extend(e_top3.cpu().tolist())

changed = sum(
    d != e
    for d, e in zip(deberta_top3, ensemble_top3)
)

print("Rows with changed ordered Top-3:", changed)

Rows with changed ordered Top-3: 41


In [21]:
import torch

label_map = ["A", "B", "C", "D", "E"]

top3_predictions = []

with torch.no_grad():

    for dbatch, rbatch in zip(dloader, rloader):

        dbatch = {k: v.to(device) for k, v in dbatch.items()}
        rbatch = {k: v.to(device) for k, v in rbatch.items()}

        d_probs = torch.softmax(deberta(**dbatch).logits, dim=1)
        r_probs = torch.softmax(roberta(**rbatch).logits, dim=1)

        ensemble_probs = (0.3 * d_probs) + (0.7 * r_probs)

        top3 = torch.topk(ensemble_probs, k=3, dim=1).indices

        for row in top3.cpu().tolist():
            top3_predictions.append([label_map[i] for i in row])

In [22]:
true_labels = train["answer"].iloc[:100].tolist()

map3 = 0.0

for pred, truth in zip(top3_predictions, true_labels):
    if truth == pred[0]:
        map3 += 1.0
    elif truth == pred[1]:
        map3 += 1/2
    elif truth == pred[2]:
        map3 += 1/3

map3 /= len(true_labels)

print(f"MAP@3 = {map3:.6f}")

MAP@3 = 0.913333
